# Datamine WIREFILL Process Example

This notebook demonstrates how to configure and run the **`wirefill`** process wrapper in `dmstudio`.

## Process Description

## WIREFILL

**Note** : This is a _superprocess_ and running it may have an effect on other Datamine files in the project.

**Note** : This process supports **[retrieval criteria](<../COMMON/Retrieval_Criteria_Overview.md>)**.

This process generates a block model from a wireframe model.

The optional input block model prototype file (**PROTO**) may be either an existing block model or a block model prototype file and can be regular or rotated.

The process only uses the model extent and cell size fields as specified in the data definition - any actual cells/subcells and their associated data columns are ignored. If a prototype model is not specified, then the model dimensions are calculated automatically from the parameter described below and the extent of the wireframe points data. The input wireframe file (**WIRETR** , **WIREPT**) may consist of one or more solid wireframes, or one or more single surface DTMs. It may not contain both solid wireframes and DTMs. An optional attribute field can be defined (**ZONE**) and added to the output model file (**MODEL**). If this field exists in the wireframe triangle file then the wireframe attribute values are passed from the wireframes to the block model cells.

The **CELLXMIN** , **CELLYMIN** and **CELLZMIN** parameters define the minimum cell size in the X, Y and Z directions. If set to zero then seam filling is used, that is, cells are split once at the wireframe boundary. Only one of the values **CELLXMIN** , **CELLYMIN** , and **CELLZMIN** may be zero.

The following parameters are optional:

* A zone code parameter (**ZCODE**) can be defined for insertion in the output block model ZONE field. This parameter is ignored if the field ZONE exists in the wireframe triangle file.

* The **CELLXMAX** , **CELLYMAX** and **CELLZMAX** parameters define the maximum (ie parent) cell size in the X, Y and Z directions. These are ignored if a prototype block model (**PROTO**) is defined.

### Cell Creation Methods

The **WIRETYPE** parameter is used to define the type of wireframe model to be filled with cells as follows:

* **1** (Solid - create cells inside.):

* **2** (Surface - create cells below.):

* **3** (Surface - create cells above.):

* **4** (Surface - create cells to the South.):

* **5** (Surface - create cells to the North.):

* **6** (Surface - create cells to the West.):

* **7** (Surface - create cells to the East.):

### Input Files:

* **proto** (Block Model Prototype File):
  Model prototype file. You may specify the name of an existing block model to define the
  model prototype settings. The process only uses the model extent and cell size fields as
  specified in the data definition - any actual cells or subcells are ignored. If a
  prototype model is not specified, then the model dimensions are calculated automatically
  from the parameter described below and the extent of the wireframe points data.
  Required=No

* **wiretr** (Wireframe Triangle):
  Input wireframe triangle file. The wireframe may consist of one or more solid
  wireframes, or one or more single surface DTMs. It may not contain both solid wireframes
  and DTMs.
  Required=Yes

* **wirept** (Wireframe Points):
  Input wireframe points file.
  Required=Yes

### Output Files:

* **model** (Block Model File):
  Output block model file. This will include the 13 standard model fields plus the
  **ZONE** field, if specified.
  Required=Yes

### Fields:

* **zone** (Any : Undefined):
  Name of the attribute field to be created in the output model file. If this field
  exists in the wireframe file then the wireframe attribute values are passed from the
  wireframes to the model cells.
  Default=Undefined
  Required=Yes

### Parameters:

* **zcode**:
  Zone code to be inserted in the output model **ZONE** field. This parameter is ignored
  if the field **ZONE** exists in the wireframe triangle file.
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **wiretype**:
  Type of wireframe model to be filled with cells. Select one of the following options,
  with the default being 1. See **Cell Creation Methods**.
  Range=1,7
  Values=1,2,3,4,5,6,7
  Default=1
  Required=Yes

* **cellxmin**:
  Minimum cell size in the X direction. If it is set to zero then seam filling is used -
  ie the cell is split once at the wireframe boundary. Only one of the values **CELLXMIN**
  , **CELLYMIN** , and **CELLZMIN** may be zero.
  Range=0,10000
  Values=Undefined
  Default=2.5
  Required=Yes

* **cellxmax**:
  Maximum (ie parent) cell size in the X direction. This is ignored if **PROTO** is
  defined.
  Range=0.5,10000
  Values=Undefined
  Default=10
  Required=No

* **cellymin**:
  Minimum cell size in the Y direction. If it is set to zero then seam filling is used -
  ie the cell is split once at the wireframe boundary. Only one of the values **CELLXMIN**
  , **CELLYMIN** , and **CELLZMIN** may be zero.
  Range=0,10000
  Values=Undefined
  Default=2.5
  Required=Yes

* **cellymax**:
  Maximum (ie parent) cell size in the Y direction. This is ignored if **PROTO** is
  defined.
  Range=0.5,10000
  Values=Undefined
  Default=10
  Required=No

* **cellzmin**:
  Minimum cell size in the Z direction. If it is set to zero then seam filling is used -
  ie the cell is split once at the wireframe boundary. Only one of the values **CELLXMIN**
  , **CELLYMIN** , and **CELLZMIN** may be zero.
  Range=0,10000
  Values=Undefined
  Default=2.5
  Required=Yes

* **cellzmax**:
  Maximum (parent) cell size in the Z direction. This is ignored if **PROTO** is defined.
  Range=0.5,1000
  Values=Undefined
  Default=10
  Required=No

* **checkrot**:
  Set to 1 to automatically detect and correctly process rotated models. Using this
  parameter means that the input wireframe points file no longer needs to be transformed
  into the model space before using this process.
  Range=0, 1
  Values=Undefined
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('wirefill')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically (4 levels up from subfolders)
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute wirefill
print("Running wirefill...")
dm_cmd.wirefill(
    wiretr_i='t_SurfaceTriangles',  # required
    wirept_i='t_SurfaceTriangles',  # required
    model_o='t_wirefill_out',  # required
    zone_f='optional',  # required
    zcode_p=1,  # required
    wiretype_p='required_val',  # required
    cellxmin_p='required_val',  # required
    cellymin_p='required_val',  # required
    cellzmin_p='required_val',  # required
    # proto_i='t_mod1',  # optional
    # cellxmax_p=10,  # optional
    # cellymax_p=10,  # optional
    # cellzmax_p=10,  # optional
    # checkrot_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("wirefill execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_wirefill_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")